In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# Offline (RTX / L4, no internet) Clifford Flow runner. The code comes from a repo snapshot
# dataset (git archive of the branch, see pose3d/USAGE.md "Kaggle"), the libraries from
# syfry5suvzovvakmuj/ga-research-offline-deps, W&B runs offline (gpu_pool/rtx_monitor.py replays the
# WANDB_SYNC log lines, or read them from `kaggle kernels logs`).
#
# Reference recipe = main's defaults, run clifford_flow_gatr_warp_synth_ema_b64 (kernel
# clifford-gatr-ema-b64-rtx): GATr vector field + Clifford condition head, real PASCAL3D+ with warp +
# RenderForCNN renders from the RAM pack syfry5suvzovvakmuj/pascal3d-synth-pack, EMA weights, fixed
# validation noise, global batch 64 with lr scaled by sqrt(64/32). 32-sample medoid: 8.95 deg
# class-mean median, 7.85 deg median. BEST_SETUP spells those flags out (so an older snapshot with other
# defaults still runs the same recipe); EXTRA holds what this run changes.
#
# Attach: syfry5suvzovvakmuj/pascal3d, pascal3d-ram-cache, pascal3d-synth-pack,
# ga-research-offline-deps, a ga-research-gatr-deps copy (einops + gatr wheels) and ONE repo snapshot.
# The account pushing the kernel must be able to see every one of them (private datasets: own copies).
BEST_SETUP = ['--vector_field', 'gatr', '--use_warp', '--use_synth', '--num_workers', '24', '--ema',
              '--fixed_val_noise', '--batch_size', '64', '--lr_scaling', 'sqrt']
RUN_NAME = 'clifford_flow_gatr_x1v_seed1'
EXTRA = ['--flow_param', 'x1', '--x1_loss', 'velocity', '--seed', '1']   # x1 output, reference loss
SNAPSHOT_MARKER = '"tangent", "geodesic", "velocity"'   # the snapshot's config.py must contain it (stale-snapshot guard)
ARGS = BEST_SETUP + EXTRA

In [ ]:
import glob, os, shutil, subprocess, tarfile

# The snapshot dataset is either auto-extracted by Kaggle (a GA-Research/ tree) or still a tar.gz.
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
if markers:
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    repo_root = markers[0][:markers[0].index('/pose3d/pose3d/config.py')]
    shutil.copytree(repo_root, '/kaggle/working/GA-Research')
else:
    tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
print(sorted(os.listdir('/kaggle/working/GA-Research')))
config_src = open('/kaggle/working/GA-Research/pose3d/pose3d/config.py').read()
assert SNAPSHOT_MARKER in config_src, f'stale snapshot: {SNAPSHOT_MARKER!r} is not in config.py'

In [ ]:
# Libraries Kaggle lacks, from the offline wheels (no index, no deps: the image's torch stays).
deps = next(p for p in glob.glob('/kaggle/input/**/ga-research-offline-deps', recursive=True)
            + glob.glob('/kaggle/input/ga-research-offline-deps'))
wheels = sorted(glob.glob(os.path.join(deps, '*.whl')))
# GATr (the reference vector field) and einops: from any mounted *ga-research-gatr-deps* copy.
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
wheels += sorted(w for d in gatr_deps[:1] for w in glob.glob(os.path.join(d, '*.whl')))
print('wheels:', [os.path.basename(w) for w in wheels])
subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)

# torchvision's resnet101 DEFAULT weights, so the pretrained backbone needs no download.
ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
shutil.copy(os.path.join(deps, 'resnet101-cd907fc2.pth'), os.path.join(ckpt_dir, 'resnet101-cd907fc2.pth'))

import importlib, sys
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford', 'opt_einsum']:
    importlib.import_module(mod)
if '--vector_field' in ARGS and ARGS[ARGS.index('--vector_field') + 1] == 'gatr':
    sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
    from pose3d.models.gatr_denoiser import _import_gatr
    print('GATr:', _import_gatr())   # fails here, not hours in, when the gatr-deps copy is missing
print('python', sys.version.split()[0], '| all offline deps import cleanly')

In [ ]:
# No internet: wandb writes local files only; the trainer also prints WANDB_SYNC lines per epoch.
os.environ['WANDB_MODE'] = 'offline'

In [ ]:
import sys, os
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
PASCAL = next(p for p in ['/kaggle/input/datasets/syfry5suvzovvakmuj/pascal3d', '/kaggle/input/pascal3d']
              if os.path.exists(p))
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={PASCAL}', '--platform=kaggle',
       f'--run_name={RUN_NAME}', *ARGS]
print(' '.join(cmd))
# check=True so a crash marks the Kaggle run as failed instead of passing silently.
subprocess.run(cmd, check=True, env={**os.environ})